In [15]:
import numpy as np
import os
import re

def load_viewpoint_poses(folder_path):
    """
    Loads all .npy pose files from a directory into a list, 
    sorted numerically by the index in the filename.
    """
    # Regex to capture the index number from 'viewpoint_pose_X.npy'
    def extract_number(filename):
        match = re.search(r'viewpoint_pose_(\d+)\.npy', filename)
        return int(match.group(1)) if match else -1

    # Filter for .npy files and sort them (0, 1, 2... instead of 0, 1, 10...)
    npy_files = [f for f in os.listdir(folder_path) if f.endswith('.npy')]
    npy_files.sort(key=extract_number)

    # Load data into list
    viewpoint_poses = []
    for file_name in npy_files:
        full_path = os.path.join(folder_path, file_name)
        try:
            pose = np.load(full_path)
            viewpoint_poses.append(pose)
        except Exception as e:
            print(f"Error loading {file_name}: {e}")

    return viewpoint_poses

# Usage
path = r'C:\Users\Alvan\Documents\Alvan\Data\Code\Python\Main\anr_pkg\src\viewpoints_candidate'
viewpoint_poses = load_viewpoint_poses(path)

print(f"Loaded {len(viewpoint_poses)} matrices.")
if viewpoint_poses:
    print("Example of first matrix:\n", viewpoint_poses[0])

Loaded 27 matrices.
Example of first matrix:
 [[         -1  3.0683e-16 -3.2643e-16  1.6405e-13]
 [ 1.2246e-16     0.88811     0.45963        -164]
 [ 4.3094e-16     0.45963    -0.88811      413.58]
 [          0           0           0           1]]


In [19]:
import open3d as o3d
import numpy as np

def visualize_transform(matrix, axis_size=50.0):
    """
    Visualizes a 4x4 transformation matrix as a coordinate frame.
    Red = X-axis, Green = Y-axis, Blue = Z-axis.
    """
    # 1. Create a coordinate frame to represent the pose
    # 'size' determines the length of the axis arrows
    pose_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(
        size=axis_size, origin=[0, 0, 0]
    )
    
    # 2. Apply the transformation matrix to the frame
    pose_frame.transform(matrix)
    
    # 3. Create a static world frame at the origin for reference
    world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(
        size=axis_size/2, origin=[0, 0, 0]
    )
    
    print(f"Visualizing Pose:\nTranslation: {matrix[:3, 3]}")
    
    # 4. Draw geometries
    o3d.visualization.draw_geometries(
        [pose_frame, world_frame, mesh],
        window_name="Pose Visualization (World=Small, Transform=Large)",
        width=1024, height=768
    )

SOURCE_PATH = "workpiece/first_object.stl" #  CAD model
mesh = o3d.io.read_triangle_mesh(SOURCE_PATH)
mesh.compute_vertex_normals() # Makes the shading look better

visualize_transform(viewpoint_poses[0])

Visualizing Pose:
Translation: [ 1.6405e-13        -164      413.58]
